In [43]:
# DS4400 HW 3
# Eunchae Hong
# Problem 1: Logistic Regression

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score
)

# load the spam database in
spam_data = pd.read_csv("spambase/spambase.data", header = None)

In [71]:
''' 
    Part 1 
    Split the original data into 75% for training and 25% for testing.
    Choose the training set at random
    Train a logistic regression model on the training set and output the following **on the testing set**:
        1. Confusion matrix
        2. Accuracy, Error
        3. Precision, Recall, F1 score
'''
# get the X and y values for the train_test_split
X = spam_data.iloc[:, :-1].values
y = spam_data.iloc[:, -1].values

# set the test_size at 0.25 to have 25% for testing, and random state set at 5 to get the same split every time
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 5)
model = LogisticRegression(max_iter = 10000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix\n", cm)

# accuracy, error, precision, recall, and F1 score
accuracy = accuracy_score(y_test, y_pred)
error = 1 - accuracy
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {round(accuracy, 3)}\nError: {round(error, 3)}")
print(f"Precision: {round(precision, 3)}\nRecall: {round(recall, 3)}\nF1 Score: {round(f1, 3)}")

Confusion Matrix
 [[672  36]
 [ 46 397]]
Accuracy: 0.929
Error: 0.071
Precision: 0.917
Recall: 0.896
F1 Score: 0.906


In [70]:
'''
    Part 2
    Print the coefficients of the features in the model. Which features contribute mostly to the prediction? Which ones are positively
    correlated and which ones are negatively correlated with the SPAM class? (answered in the text box below)
'''
# grab the feature names from spam.names
with open("spambase/spambase.names", "r") as file:
    lines = file.readlines()

feature_names = []
for line in lines:
    if "continuous" in line and "|" not in line:
        feature_names.append(line.split(":")[0].strip())

# get coefficients from model above and put in a DF
coeffs = model.coef_[0]
coef_df = pd.DataFrame({"Feature": feature_names, "Coefficient": coeffs
}).sort_values("Coefficient", ascending = False)

# print the coefficients of the features in the model
print(coef_df)

# get top 3 and bottom 3 features to see which contributed the most and least
print("Top 3\n", coef_df.head(3).to_string(index = False))
print("Bottom 3\n", coef_df.tail(3).to_string(index = False))

                       Feature  Coefficient
6             word_freq_remove     1.262354
15              word_freq_free     1.205763
22               word_freq_000     1.069854
4                word_freq_our     0.946965
16          word_freq_business     0.811631
52                 char_freq_$     0.766931
7           word_freq_internet     0.698394
23             word_freq_money     0.541733
5               word_freq_over     0.540630
17             word_freq_email     0.477366
19            word_freq_credit     0.458467
51                 char_freq_!     0.412755
8              word_freq_order     0.338229
20              word_freq_your     0.328478
3                 word_freq_3d     0.290209
14         word_freq_addresses     0.260457
53                 char_freq_#     0.240658
12            word_freq_people     0.194682
10           word_freq_receive     0.173085
2                word_freq_all     0.153056
9               word_freq_mail     0.143451
21              word_freq_font  

***Part 2 Questions: Which features contribute mostly to the prediction? Which ones are positively correlated and which ones are negatively correlated with the SPAM class?***

The features that contribute mostly to the prediction are "word_freq_remove", "word_freq_free", and "word_freq_000" as they have the highest positive coefficients, which makes them positvely correlated with the SPAM class. Therefore the more times words like "remove", "free", or "000" were included in an email they were positivley correlated to the SPAM class. On the other hand the features that were the most negatively correlated with the SPAM class were "word_freq_edu", "word_freq_hp", and "word_freq_george", with the highest negative coefficients. Therefore the more times words like "edu", "hp", and "george" were included in an email they were negatively correlated with the SPAM class.

In [ ]:
''' Part 3
    Vary the decision threshold T{0.25,0.5,0.75,0.9} and report for each value the model accuracy, precision, and recall. 
    Comment on how these metrics vary with the choice of threshold.
'''
# set the given decision thresholds
thresholds = [0.25, 0.5, 0.75, 0.9]

# get the probabilities for the spam emails, using column 1 to look at prob(spam)
y_prob = model.predict_proba(X_test)[:, 1]

# using a for loop iterate between each threshold
for num in thresholds:
    y_pred_T = (y_prob >= num).astype(int)
    accuracy = accuracy_score(y_test, y_pred_T)
    precision = precision_score(y_test, y_pred_T)
    recall = recall_score(y_test, y_pred_T)

    print(f"For threshold = {num}\nAccuracy: {round(accuracy, 3)}\nPrecision: {round(precision, 3)}\nRecall:{round(recall, 3)}")

For threshold = 0.25
Accuracy: 0.877
Precision: 0.774
Recall:0.959
For threshold = 0.5
Accuracy: 0.92
Precision: 0.896
Recall:0.896
For threshold = 0.75
Accuracy: 0.885
Precision: 0.921
Recall:0.767
For threshold = 0.9
Accuracy: 0.821
Precision: 0.951
Recall:0.564


***Part 3 Questions: Comment on how these metrics vary with the choice of threshold***

As the threshold increases, the recall significantly decreases, the precision slightly increases, and the accuracy initially increases, but then goes down. Additionally at threshold = 0.5 the precision and recall are equal. Using these metrics it helps us decide which threshold works best for us based on the goals of our model.